In [79]:
# ============================================
# CALCULATOR USING FLEX AND BISON
# Google Colab - Single Cell
# ============================================

# 1. Install FLEX, BISON and GCC
!apt-get update -qq
!apt-get install -y flex bison gcc -qq

# 2. Create cal.l
with open("cal.l", "w") as f:
    f.write(r'''
%{
#include "cal.tab.h"
#include <stdlib.h>
%}

DIGIT [0-9]+(\.[0-9]+)?

%option noyywrap

%%

{DIGIT} {
    yylval.val = atof(yytext);
    return NUM;
}

[ \t] {
}

\n {
    return '\n';
}

. {
    return yytext[0];
}

%%
''')

# 3. Create cal.y
with open("cal.y", "w") as f:
    f.write(r'''
%{
#include <stdio.h>
#include <stdlib.h>

int yylex(void);
int yyerror(char *s);
%}

%union {
    double val;
}

%token <val> NUM

%type <val> E

%left '+' '-'
%left '*' '/'
%right UMINUS

%%

statement:
    E '\n'
    {
        printf("Answer: %g\n", $1);
    }
    ;

E:
      E '+' E
      {
          $$ = $1 + $3;
      }

    | E '-' E
      {
          $$ = $1 - $3;
      }

    | E '*' E
      {
          $$ = $1 * $3;
      }

    | E '/' E
      {
          if ($3 == 0)
          {
              printf("Error: Division by zero\n");
              $$ = 0;
          }
          else
          {
              $$ = $1 / $3;
          }
      }

    | '(' E ')'
      {
          $$ = $2;
      }

    | '-' E %prec UMINUS
      {
          $$ = -$2;
      }

    | NUM
      {
          $$ = $1;
      }
    ;

%%

int main()
{
    printf("Enter the expression:\n");
    yyparse();
    return 0;
}

int yyerror(char *s)
{
    printf("Invalid expression: %s\n", s);
    return 0;
}
''')

# 4. Remove old generated files
!rm -f cal.tab.c cal.tab.h lex.yy.c calc

# 5. Generate BISON and FLEX files
!bison -d cal.y
!flex cal.l

# 6. Compile
!gcc lex.yy.c cal.tab.c -o calc -lfl

# 7. Give input
with open("input.txt", "w") as f:
    f.write("2+2\n")

# 8. Execute
import subprocess

result = subprocess.run(
    ["./calc"],
    stdin=open("input.txt", "r"),
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE,
    text=True
)

print(result.stdout)

if result.stderr:
    print(result.stderr)

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Enter the expression:
Answer: 4

